In [ ]:
import os
import math
import copy
import random
import numpy as np
import pandas as pd
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

train_csv = "/content/train.csv"
val_csv   = "/content/val.csv"
test_csv  = "/content/test.csv"

df_train = pd.read_csv(train_csv)
df_val   = pd.read_csv(val_csv)
df_test  = pd.read_csv(test_csv)

cond_cols = ["Frequency (Hz)", "Storage modulus (Pa)", "Loss modulus (Pa)"]

all_cols = df_train.columns.tolist()
gen_cols = [c for c in all_cols if c not in cond_cols]

print("Condition cols:", cond_cols)
print("Generated cols :", gen_cols)


# for c in cond_cols + gen_cols:
#     df_train[c] = pd.to_numeric(df_train[c], errors="coerce")
#     df_val[c]   = pd.to_numeric(df_val[c], errors="coerce")
#     df_test[c]  = pd.to_numeric(df_test[c], errors="coerce")

# df_train = df_train.dropna(subset=cond_cols + gen_cols).reset_index(drop=True)
# df_val   = df_val.dropna(subset=cond_cols + gen_cols).reset_index(drop=True)
# df_test  = df_test.dropna(subset=cond_cols).reset_index(drop=True)

print("Train size:", len(df_train))
print("Val size  :", len(df_val))
print("Test size :", len(df_test))


with open('/content/X_scaler_cvae.pkl', 'rb') as f:
  x_scaler = pickle.load(f)

with open('/content/Y_scaler_cvae.pkl', 'rb') as f:
  y_scaler = pickle.load(f)

X_train = x_scaler.transform(df_train[gen_cols]).astype(np.float32)
Y_train = y_scaler.transform(df_train[cond_cols]).astype(np.float32)

X_val = x_scaler.transform(df_val[gen_cols]).astype(np.float32)
Y_val = y_scaler.transform(df_val[cond_cols]).astype(np.float32)

Y_test = y_scaler.transform(df_test[cond_cols]).astype(np.float32)

x_dim = X_train.shape[1]
y_dim = Y_train.shape[1]

print("x_dim =", x_dim, "y_dim =", y_dim)

X_train = torch.tensor(X_train, dtype=torch.float32)
Y_train = torch.tensor(Y_train, dtype=torch.float32)

X_val = torch.tensor(X_val, dtype=torch.float32)
Y_val = torch.tensor(Y_val, dtype=torch.float32)


train_ds = TensorDataset(X_train, Y_train)
val_ds   = TensorDataset(X_val, Y_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=False)


def make_beta_schedule(T, beta_start=1e-4, beta_end=2e-2):
  return torch.linspace(beta_start, beta_end, T, dtype=torch.float32)

T = 1000
betas = make_beta_schedule(T).to(device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

alpha_bars_prev = torch.cat([torch.tensor([1.0], device=device), alpha_bars[:-1]], dim=0)

posterior_variance = betas * (1.0 - alpha_bars_prev) / (1.0 - alpha_bars)
posterior_variance[0] = 1e-8

def extract(a, t, x_shape):
  out = a.gather(0, t)
  return out.view(-1, 1).expand(x_shape)

class SinusoidalTimeEmbedding(nn.Module):
  def __init__(self, dim):
    super().__init__()
    self.dim = dim

  def forward(self, t):

    half_dim = self.dim // 2
    emb_scale = math.log(10000) / max(half_dim - 1, 1)
    emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb_scale)
    emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if self.dim % 2 == 1:
      emb = F.pad(emb, (0, 1))
    return emb

class ConditionalDenoiser(nn.Module):
  def __init__(self, x_dim, y_dim, hidden_dim=256, time_dim=64, dropout=0.1):
    super().__init__()

    self.time_emb = SinusoidalTimeEmbedding(time_dim)

    self.net = nn.Sequential(
            nn.Linear(x_dim + y_dim + time_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, x_dim)
        )

  def forward(self, x_t, t, y):
    t_emb = self.time_emb(t)
    inp = torch.cat([x_t, y, t_emb], dim=1)
    return self.net(inp)

model = ConditionalDenoiser(
    x_dim=x_dim,
    y_dim=y_dim,
    hidden_dim=256,
    time_dim=64,
    dropout=0.1
).to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=8,
    threshold=1e-4,
    min_lr=1e-6
)
def evaluate(model, loader):
  model.eval()
  total_loss = 0.0
  total_samples = 0
  with torch.no_grad():
    for x0, y in loader:
      x0 = x0.to(device)
      y = y.to(device)

      B = x0.size(0)
      t = torch.randint(0, T, (B,), device=device)
      t = t.sort()[0]
      noise = torch.randn_like(x0)

      sqrt_ab = extract(sqrt_alpha_bars, t, x0.shape)
      sqrt_1mab = extract(sqrt_one_minus_alpha_bars, t, x0.shape)

      x_t = sqrt_ab * x0 + sqrt_1mab * noise
      noise_pred = model(x_t, t, y)

      loss = F.mse_loss(noise_pred, noise, reduction="mean")
      total_loss += loss.item() * B
      total_samples += B

  return total_loss / total_samples


EPOCHS = 500
PATIENCE = 20

best_state = None
best_val_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0

for epoch in range(EPOCHS):
  model.train()
  running_train_loss = 0.0

  for x0, y in train_loader:
    x0 = x0.to(device)
    y = y.to(device)

    B = x0.size(0)
    t = torch.randint(0, T, (B,), device=device).long()
    noise = torch.randn_like(x0)

    sqrt_ab = extract(sqrt_alpha_bars, t, x0.shape)
    sqrt_1mab = extract(sqrt_one_minus_alpha_bars, t, x0.shape)

    x_t = sqrt_ab * x0 + sqrt_1mab * noise
    noise_pred = model(x_t, t, y)

    loss = F.mse_loss(noise_pred, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_train_loss += loss.item() * B

  train_loss = running_train_loss / len(train_ds)
  val_loss = evaluate(model, val_loader)
  scheduler.step(val_loss)

  if val_loss < best_val_loss:
    best_val_loss = val_loss
    best_epoch = epoch + 1
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
  else:
    epochs_no_improve += 1

  if (epoch + 1) % 25 == 0 or epoch == 0:
    print(
            f"Epoch {epoch+1:4d}/{EPOCHS} | "
            f"train_loss = {train_loss:.6f} | "
            f"val_loss = {val_loss:.6f} | "
            f"best_val = {best_val_loss:.6f} @ epoch {best_epoch}"
        )

  if epochs_no_improve >= PATIENCE:
    print(f"Early stopping triggered at epoch {epoch+1}")
    break

print("\nBest validation loss:", best_val_loss)
print("Best epoch:", best_epoch)

model.load_state_dict(best_state)

# ckpt_path = "/content/best_conditional_tabddpm.pt"
# torch.save({
#     "model_state_dict": model.state_dict(),
#     "x_scaler_mean": x_scaler.mean_,
#     "x_scaler_scale": x_scaler.scale_,
#     "y_scaler_mean": y_scaler.mean_,
#     "y_scaler_scale": y_scaler.scale_,
#     "cond_cols": cond_cols,
#     "gen_cols": gen_cols,
#     "all_cols": all_cols,
#     "best_val_loss": best_val_loss,
#     "best_epoch": best_epoch,
# }, ckpt_path)
# print("Saved checkpoint to:", ckpt_path)


def sample_conditional(model, y_cond, num_steps=T):

  model.eval()
  N = y_cond.size(0)

  x_t = torch.randn(N, x_dim, device=device)
  with torch.no_grad():
    for step in reversed(range(num_steps)):
      t = torch.full((N,), step, device=device, dtype=torch.long)

      beta_t = extract(betas, t, x_t.shape)
      sqrt_one_minus_ab_t = extract(sqrt_one_minus_alpha_bars, t, x_t.shape)
      sqrt_recip_alpha_t = extract(sqrt_recip_alphas, t, x_t.shape)

      eps_theta = model(x_t, t, y_cond)

      model_mean = sqrt_recip_alpha_t * (
            x_t - (beta_t / sqrt_one_minus_ab_t) * eps_theta
        )

      if step > 0:
        var_t = extract(posterior_variance, t, x_t.shape)
        noise = torch.randn_like(x_t)
        x_t = model_mean + torch.sqrt(var_t) * noise
      else:
        x_t = model_mean

  return x_t

Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32, device=device)

X_gen_std = sample_conditional(model, Y_test_tensor, num_steps=T).cpu().numpy()

X_gen = x_scaler.inverse_transform(X_gen_std)

x_min = df_train[gen_cols].min().values.astype(np.float32)
x_max = df_train[gen_cols].max().values.astype(np.float32)
X_gen = np.clip(X_gen, x_min, x_max)


synth = pd.DataFrame(X_gen, columns=gen_cols)

for c in cond_cols:
  synth[c] = df_test[c].values

synth = synth[all_cols]

out_csv = "/content/synth_tabddpm.csv"
synth.to_csv(out_csv, index=False)

Using device: cuda
Condition cols: ['Frequency (Hz)', 'Storage modulus (Pa)', 'Loss modulus (Pa)']
Generated cols : ['Acylamide Conc. %', 'Bis-acrylamide conc %', 'Photo-initiator conc. %', 'Layer Height. (micron)', 'Bottom Layer exposure time (s) ', 'Exposure time (s)']
Train size: 1868
Val size  : 234
Test size : 234
x_dim = 6 y_dim = 3
Epoch    1/500 | train_loss = 0.585564 | val_loss = 0.307875 | best_val = 0.307875 @ epoch 1
Epoch   25/500 | train_loss = 0.204614 | val_loss = 0.191375 | best_val = 0.151799 @ epoch 24
Epoch   50/500 | train_loss = 0.169398 | val_loss = 0.126908 | best_val = 0.115128 @ epoch 48
Early stopping triggered at epoch 68

Best validation loss: 0.11512811552000861
Best epoch: 48
